In [5]:
# CELDA 1 - Conexión
from zeep import Client

wsdl = "https://servayto.madrid.es/MTPAR_WSINFO/InfoParking?wsdl"
client = Client(wsdl=wsdl)

In [6]:
# CELDA 2 - Operaciones disponibles
for service in client.wsdl.services.values():
    for port in service.ports.values():
        for operation in port.binding._operations.values():
            print(operation.name)

GetListParking
GetListFeatures
GetListStreetPoisParking
GetInfoParkingPoisForCoordinate
GetDetailParking


In [7]:
# CELDA 3 - Investigar GetListParking
print(client.service.GetListParking.__doc__)

GetListParking(language: xsd:string) -> GetListParkingResult: ns2:responseListParking


In [8]:
from zeep import Client, Settings

settings = Settings(strict=False)

wsdl = "https://servayto.madrid.es/MTPAR_WSINFO/InfoParking?wsdl"

client = Client(
    wsdl=wsdl,
    settings=settings
)

print("Cliente creado")

Cliente creado


In [9]:
resultado = client.service.GetListParking(language="ES")

print(resultado)

Fault: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not prepare statement

In [10]:
from lxml import etree

raw = resultado["_raw_elements"][0]

print(
    etree.tostring(
        raw,
        pretty_print=True,
        encoding="unicode"
    )[:5000]
)

NameError: name 'resultado' is not defined

In [ ]:
import pandas as pd

raw = resultado["_raw_elements"][0]

parkings = []

for parking in raw:
    datos = {}

    for campo in parking:
        nombre_campo = campo.tag.split("}")[-1]

        if len(campo) == 0:
            datos[nombre_campo] = campo.text
        else:
            for subcampo in campo:
                nombre_subcampo = subcampo.tag.split("}")[-1]
                datos[f"{nombre_campo}_{nombre_subcampo}"] = subcampo.text

    parkings.append(datos)

df = pd.DataFrame(parkings)

df.head()

,address,administrativeArea,areaCode,category,country,family,familyCode,id,latitude,longitude,name,nickName,state,town,type,lstOccupation_occupation
0,Calle Corazón de María,Madrid,28002,POI Categoria Parking,España,POI Familia,001,3,40.438524,-3.645525,Corazón de María II,MaríaII,Madrid,Madrid,POI Tipo Parking,NaN
1,Plaza Encuentro,Madrid,28030,POI Categoria Parking,España,POI Familia,001,4,40.405464,-3.651354,Encuentro,Encuentro,Madrid,Madrid,POI Tipo Parking,NaN
2,Calle de la Hiedra,Madrid,28036,POI Categoria Parking,España,POI Familia,001,5,40.472181,-3.679160,Nuestra Señora del Recuerdo,Recuerdo,Madrid,Madrid,POI Tipo Parking,NaN
3,Calle Perez de Victoria,Madrid,28023,POI Categoria Parking,España,POI Familia,001,6,40.4568,-3.7832,Corona Boreal,C.Boreal,Madrid,Madrid,POI Tipo Parking,NaN
4,"Avda. de Portugal, s/n. Frente al nº 51",Madrid,28011,POI Categoria Parking,España,POI Familia,001,7,40.415415,-3.727515,Avenida de Portugal,AvPortugal,Madrid,Madrid,POI Tipo Parking,NaN


In [ ]:
df.shape

(75, 16)

In [ ]:
df.columns.tolist()

['address',
 'administrativeArea',
 'areaCode',
 'category',
 'country',
 'family',
 'familyCode',
 'id',
 'latitude',
 'longitude',
 'name',
 'nickName',
 'state',
 'town',
 'type',
 'lstOccupation_occupation']

In [ ]:
df[["name", "lstOccupation_occupation"]].head(10)

,name,lstOccupation_occupation
0,Corazón de María II,NaN
1,Encuentro,NaN
2,Nuestra Señora del Recuerdo,NaN
3,Corona Boreal,NaN
4,Avenida de Portugal,NaN
5,Paseo de Recoletos,NaN
6,Condesa de Gavia,NaN
7,Vázquez de Mella,NaN
8,Almagro,NaN
9,Jacinto Benavente,NaN


In [ ]:
type(df.loc[0, "lstOccupation_occupation"])

numpy.float64

In [ ]:
for service in client.wsdl.services.values():
    for port in service.ports.values():
        operations = port.binding._operations
        
        print(operations["GetDetailParking"])

GetDetailParking(parametersDetailParking: ns2:paramDetailParking) -> GetDetailParkingResult: ns2:responseParkingDetail


In [ ]:
tipo = client.get_type("ns2:paramDetailParking")
print(tipo)

paramDetailParking({http://schemas.datacontract.org/2004/07/InfoParking}paramDetailParking(date: xsd:dateTime, family: xsd:string, id: xsd:int, language: xsd:string, publicData: xsd:boolean))


In [ ]:
from datetime import datetime

parametros = {
    "date": datetime.now(),
    "family": "001",
    "id": 3,
    "language": "ES",
    "publicData": True
}

detalle = client.service.GetDetailParking(
    parametersDetailParking=parametros
)

print(detalle)

{
    'code': 0,
    'message': 'Operación registrada OK.',
    'Data': None,
    '_raw_elements': deque([<Element {http://schemas.datacontract.org/2004/07/InfoParking}ArrayOfparkingDetails at 0x1beb3c9ff00>])
}


In [ ]:
from lxml import etree

raw_detalle = detalle["_raw_elements"][0]

print(
    etree.tostring(
        raw_detalle,
        pretty_print=True,
        encoding="unicode"
    )[:8000]
)

<ns2:ArrayOfparkingDetails xmlns:ns2="http://schemas.datacontract.org/2004/07/InfoParking" xmlns="http://tempuri.org/" xmlns:ns3="http://schemas.microsoft.com/2003/10/Serialization/Arrays" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <ns2:parkingDetails>
    <ns2:family>POI Familia</ns2:family>
    <ns2:familyCode>001</ns2:familyCode>
    <ns2:general>
      <ns2:address>Calle Corazón de María</ns2:address>
      <ns2:administrativeArea>Madrid</ns2:administrativeArea>
      <ns2:areaCode>28002 </ns2:areaCode>
      <ns2:category>POI Categoria Parking</ns2:category>
      <ns2:categoryCode>001</ns2:categoryCode>
      <ns2:country>España</ns2:country>
      <ns2:icon>http://urlicono.org/neutro</ns2:icon>
      <ns2:latitude>40.438524</ns2:latitude>
      <ns2:longitude>-3.645525</ns2:longitude>
      <ns2:nickName>Corazón de María II</ns2:nickName>
      <ns2:state>Madrid  </ns2:state>
      <ns2:town>Madrid</ns2:town>
      <ns2:type>POI Tipo Parking</ns2:type>
      <ns2:

In [ ]:
parking = raw_detalle[0]

for elemento in parking:
    nombre = elemento.tag.split("}")[-1]

    if len(elemento) == 0:
        print(nombre, ":", elemento.text)
    else:
        print("\n", nombre)

        for subelemento in elemento:
            subnombre = subelemento.tag.split("}")[-1]

            if len(subelemento) == 0:
                print("  ", subnombre, ":", subelemento.text)
            else:
                print("  ", subnombre)

                for subsubelemento in subelemento:
                    subsubnombre = subsubelemento.tag.split("}")[-1]
                    print("     ", subsubnombre, ":", subsubelemento.text)

family : POI Familia
familyCode : 001

 general
   address : Calle Corazón de María
   administrativeArea : Madrid
   areaCode : 28002 
   category : POI Categoria Parking
   categoryCode : 001
   country : España
   icon : http://urlicono.org/neutro
   latitude : 40.438524
   longitude : -3.645525
   nickName : Corazón de María II
   state : Madrid  
   town : Madrid
   type : POI Tipo Parking
   typeCode : 001
id : 3

 lstFeatures
   feature
      code : 001
      content : 327
      name : Total
      nameField : Tipo plaza
      nameFieldTranslated : Número plazas
      urlIcon : http://urlicono.org/neutro
name : Corazón de María II
schedule : [24 Horas][Lunes,Martes,Miercoles,Jueves,Viernes,Sabado,Domingo][00:00][23:59]


In [ ]:
features = []

for feature in parking.find("{http://schemas.datacontract.org/2004/07/InfoParking}lstFeatures"):
    fila = {}

    for campo in feature:
        nombre = campo.tag.split("}")[-1]
        fila[nombre] = campo.text

    features.append(fila)

df_features = pd.DataFrame(features)

df_features

,code,content,name,nameField,nameFieldTranslated,urlIcon
0,001,327,Total,Tipo plaza,Número plazas,http://urlicono.org/neutro


In [ ]:
for service in client.wsdl.services.values():
    for port in service.ports.values():
        operations = port.binding._operations

        print(operations["GetListFeatures"])

GetListFeatures(parametersListFeatures: ns2:paramListFeatures) -> GetListFeaturesResult: ns2:responseListFeatures


In [ ]:
tipo_features = client.get_type("ns2:paramListFeatures")
print(tipo_features)

paramListFeatures({http://schemas.datacontract.org/2004/07/InfoParking}paramListFeatures(language: xsd:string, publicData: xsd:boolean))


In [ ]:
parametros_features = {
    "language": "ES",
    "publicData": True
}

features_resultado = client.service.GetListFeatures(
    parametersListFeatures=parametros_features
)

print(features_resultado)

{
    'code': 0,
    'message': 'Operación registrada OK.',
    'Data': None,
    '_raw_elements': deque([<Element {http://schemas.datacontract.org/2004/07/InfoParking}ArrayOffeatureType at 0x1beb3cbc800>])
}


In [ ]:
from lxml import etree

raw_features = features_resultado["_raw_elements"][0]

print(
    etree.tostring(
        raw_features,
        pretty_print=True,
        encoding="unicode"
    )[:8000]
)

<ns2:ArrayOffeatureType xmlns:ns2="http://schemas.datacontract.org/2004/07/InfoParking" xmlns="http://tempuri.org/" xmlns:ns3="http://schemas.microsoft.com/2003/10/Serialization/Arrays" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <ns2:featureType>
    <ns2:description>0</ns2:description>
    <ns2:name>General</ns2:name>
    <ns2:nameCode>001 </ns2:nameCode>
    <ns2:nameField>POI Categoria Parking</ns2:nameField>
    <ns2:nameFieldTranslated>POI Categoria Parking</ns2:nameFieldTranslated>
    <ns2:urlIcon/>
  </ns2:featureType>
  <ns2:featureType>
    <ns2:description>0</ns2:description>
    <ns2:name>Ascensor salida calle</ns2:name>
    <ns2:nameCode>007 </ns2:nameCode>
    <ns2:nameField>Tipo acceso</ns2:nameField>
    <ns2:nameFieldTranslated>Tipo acceso</ns2:nameFieldTranslated>
    <ns2:urlIcon/>
  </ns2:featureType>
  <ns2:featureType>
    <ns2:description>0</ns2:description>
    <ns2:name>Acceso con tarjeta</ns2:name>
    <ns2:nameCode>005 </ns2:nameCode>
    <ns2:

In [ ]:
features_catalogo = []

for feature in raw_features:
    fila = {}

    for campo in feature:
        nombre = campo.tag.split("}")[-1]
        fila[nombre] = campo.text

    features_catalogo.append(fila)

df_features_catalogo = pd.DataFrame(features_catalogo)

df_features_catalogo

,description,name,nameCode,nameField,nameFieldTranslated,urlIcon
0,0,General,001,POI Categoria Parking,POI Categoria Parking,None
1,0,Ascensor salida calle,007,Tipo acceso,Tipo acceso,None
2,0,Acceso con tarjeta,005,Tipo acceso,Tipo acceso,None
3,0,Tarifa por minuto Tramo 1,001,Tipo tarifa,Tipo tarifa,None
4,0,Rotacion,001,Tipo parking,Tipo parking,None
...,...,...,...,...,...,...
74,0,CCTV,026,Servicios adicionales,Servicios adicionales,None
75,0,Teléfono concesionario,005,Datos concesionario,Datos concesionario,None
76,0,Grabación de matrículas (E/S),028,Servicios adicionales,Servicios adicionales,None
77,0,Taller,010,Servicios adicionales,Servicios adicionales,None


In [ ]:
df_features_catalogo.shape

(79, 6)

In [ ]:
df_features_catalogo["name"].tolist()

['General',
 'Ascensor salida calle',
 'Acceso con tarjeta',
 'Tarifa por minuto Tramo 1',
 'Rotacion',
 'Tarifa por minuto Tramo 2',
 'Recarga eléctrica',
 'Tarifa por minuto Tramo 3',
 'Vehículo',
 'Tarifa por minuto Tramo 4',
 'Total',
 'Tarifa por minuto Tramo 5',
 'Aparcamiento',
 'Tarifa fija',
 'Lavadero',
 'Turismo',
 'Ascensor',
 'Motocicleta',
 'Abonado',
 'Autobus',
 'Reserva de plazas',
 'Bicicleta',
 'Motocicleta',
 'VMP',
 'Altura máxima',
 'Normal',
 'Disponibilidad de abonos',
 'Alta',
 'Hojas de reclamaciones',
 'Alta Bonifiacda',
 'Residente',
 'Bicicleta',
 'Residentes',
 'Reservada',
 'Teléfono incidencias',
 'Aparcamiento para bicicletas',
 'Vending',
 'Instalaciones',
 'VIA-T',
 'Mixto',
 'Normal',
 'Fax',
 'Pago desde móvil',
 'Peatonal',
 'Aseos',
 'Car sharing',
 'Ascensor para vehículos',
 'Pago en efectivo',
 'Escaleras adaptadas',
 'Cajero central presencial',
 'Eléctrica',
 'PMR',
 'Observaciones',
 'Contacto concesionario',
 'Información a usuarios',
 'Tar

In [ ]:
df[["id", "name"]].head(10)

,id,name
0,3,Corazón de María II
1,4,Encuentro
2,5,Nuestra Señora del Recuerdo
3,6,Corona Boreal
4,7,Avenida de Portugal
5,9,Paseo de Recoletos
6,10,Condesa de Gavia
7,11,Vázquez de Mella
8,12,Almagro
9,15,Jacinto Benavente


In [ ]:
from datetime import datetime

resumen_features = []

for _, fila in df[["id", "name"]].head(10).iterrows():

    parametros = {
        "date": datetime.now(),
        "family": "001",
        "id": int(fila["id"]),
        "language": "ES",
        "publicData": True
    }

    detalle_tmp = client.service.GetDetailParking(
        parametersDetailParking=parametros
    )

    raw_tmp = detalle_tmp["_raw_elements"][0]

    parking_tmp = raw_tmp[0]

    features_tmp = parking_tmp.find(
        "{http://schemas.datacontract.org/2004/07/InfoParking}lstFeatures"
    )

    numero_features = len(features_tmp) if features_tmp is not None else 0

    resumen_features.append({
        "id": fila["id"],
        "parking": fila["name"],
        "numero_features": numero_features
    })

In [ ]:
df_resumen_features = pd.DataFrame(resumen_features)

df_resumen_features

,id,parking,numero_features
0,3,Corazón de María II,1
1,4,Encuentro,1
2,5,Nuestra Señora del Recuerdo,5
3,6,Corona Boreal,1
4,7,Avenida de Portugal,5
5,9,Paseo de Recoletos,5
6,10,Condesa de Gavia,1
7,11,Vázquez de Mella,20
8,12,Almagro,5
9,15,Jacinto Benavente,5


In [ ]:
raw_lista = resultado["_raw_elements"][0]

for parking in raw_lista[:10]:
    nombre = parking.find(
        "{http://schemas.datacontract.org/2004/07/InfoParking}name"
    )

    ocupacion = parking.find(
        "{http://schemas.datacontract.org/2004/07/InfoParking}lstOccupation"
    )

    print("\nPARKING:", nombre.text if nombre is not None else "Sin nombre")

    if ocupacion is None:
        print("  Sin datos de ocupación")
    else:
        for item in ocupacion:
            for campo in item:
                campo_nombre = campo.tag.split("}")[-1]
                print(" ", campo_nombre, ":", campo.text)


PARKING: Corazón de María II
  Sin datos de ocupación

PARKING: Encuentro
  Sin datos de ocupación

PARKING: Nuestra Señora del Recuerdo
  code : 001
  free : 358
  moment : 2026-08-25T18:29:41.000+02:00
  name : Total
  renewalIndex : 0.00

PARKING: Corona Boreal
  Sin datos de ocupación

PARKING: Avenida de Portugal
  code : 001
  free : 272
  moment : 2026-08-25T18:29:58.000+02:00
  name : Total
  renewalIndex : 0.00

PARKING: Paseo de Recoletos
  code : 001
  free : 158
  moment : 2026-08-25T18:29:44.000+02:00
  name : Total
  renewalIndex : 0.00

PARKING: Condesa de Gavia
  Sin datos de ocupación

PARKING: Vázquez de Mella
  Sin datos de ocupación

PARKING: Almagro
  code : 001
  free : 371
  moment : 2026-08-25T18:29:12.000+02:00
  name : Total
  renewalIndex : 0.00

PARKING: Jacinto Benavente
  code : 001
  free : 0
  moment : 2026-08-25T18:29:00.000+02:00
  name : Total
  renewalIndex : 0.00


In [11]:
resultado_lista = client.service.GetListParking(
    language="ES"
)

print(resultado_lista)

Fault: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not prepare statement

In [12]:
resultado_lista = client.service.GetListParking(
    language="ES"
)

print(resultado_lista)

Fault: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not prepare statement

In [13]:
print(client.service)

In [14]:
parametros_features = {
    "language": "ES",
    "publicData": True
}

resultado_features = client.service.GetListFeatures(
    parametersListFeatures=parametros_features
)

print(resultado_features)

Fault: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not prepare statement

In [15]:
import requests

url = "https://servayto.madrid.es/MTPAR_WSINFO/InfoParking?wsdl"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

200
<?xml version='1.0' encoding='UTF-8'?><wsdl:definitions xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:wsdl="http://schemas.xmlsoap.org/wsdl/" xmlns:wsaw="http://www.w3.org/2006/05/addressing/wsdl" xmlns:wsam="http://www.w3.org/2007/05/addressing/metadata" xmlns:tns="http://tempuri.org/" xmlns:soap="http://schemas.xmlsoap.org/wsdl/soap/" xmlns:ns1="http://schemas.xmlsoap.org/soap/http" name="infoParking" targetNamespace="http://tempuri.org/">
  <wsdl:types>
<xs:schema xmlns:xs="http://www.w3


In [17]:
from datetime import datetime

estado_api = {}

# 1. GetListParking
try:
    client.service.GetListParking(language="ES")
    estado_api["GetListParking"] = "✅ OK"
except Exception as e:
    estado_api["GetListParking"] = f"❌ ERROR: {str(e)[:100]}"


# 2. GetListFeatures
try:
    parametros_features = {
        "language": "ES",
        "publicData": True
    }

    client.service.GetListFeatures(
        parametersListFeatures=parametros_features
    )

    estado_api["GetListFeatures"] = "✅ OK"

except Exception as e:
    estado_api["GetListFeatures"] = f"❌ ERROR: {str(e)[:100]}"


# 3. GetDetailParking
try:
    parametros_detalle = {
        "date": datetime.now(),
        "family": "001",
        "id": 3,
        "language": "ES",
        "publicData": True
    }

    client.service.GetDetailParking(
        parametersDetailParking=parametros_detalle
    )

    estado_api["GetDetailParking"] = "✅ OK"

except Exception as e:
    estado_api["GetDetailParking"] = f"❌ ERROR: {str(e)[:100]}"


# Mostrar resultados
for operacion, estado in estado_api.items():
    print(f"{operacion}: {estado}")

GetListParking: ❌ ERROR: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not p
GetListFeatures: ❌ ERROR: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not p
GetDetailParking: ❌ ERROR: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not p


### La API SOAP está accesible, pero las operaciones de datos están devolviendo temporalmente un error JDBCConnectionException del servidor.

In [18]:
for operacion, estado in estado_api.items():
    print(f"{operacion}: {estado}")

GetListParking: ❌ ERROR: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not p
GetListFeatures: ❌ ERROR: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not p
GetDetailParking: ❌ ERROR: javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not p


In [19]:
try:
    resultado = client.service.GetListParking(language="ES")
    print("✅ API FUNCIONANDO")
except Exception as e:
    print("❌ API SIGUE FALLANDO")
    print(e)

❌ API SIGUE FALLANDO
javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException: could not prepare statement
